# Lab 00 · From rules to LLM calls

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · 15–20 minute lab

## One architecture question, two ways to answer it
**Start here.** A growing social application can benefit from independent scaling and releases, but a distributed system also needs network, data, and operational support. Independent scaling and releases are benefits, but distribution adds network, data and operations costs.

Our small *screening rule* is: consider **microservices** only when independent releases are needed **and** operational prerequisites are ready; otherwise start with a **modular monolith**. `ops_ready` summarizes all four prerequisites for this example: automated CI/CD, monitoring and alerting, rapid provisioning, and teams that own operations. This screen does not prove that a service boundary or architecture is good. Team size alone never selects the architecture.

Write the rule, cover its four paths, then compare it with a single unguided LLM call.

For these four cases, the stated goal is independent releases. The lecture also motivates services by independent scaling and ownership; the two-Boolean screen is a small classroom adaptation, not a complete or universally necessary architecture test.

**At a glance**

![Lab 00 at a glance: 4 readiness cases (ops + independent releases), then Apply the rule (decide()) or Ask a model (ask() + to_label()), then Compare (agreement count).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/00.svg)

## Think about the example

A team considering independent releases needs automated delivery, monitoring, rapid provisioning, and people who can operate the services. A small application can start as a modular monolith.

**Concepts:** Independent deployment and scaling; Microservice premium; Operational prerequisites.

Use the scenario above to consider these questions before running the code:

1. Which four prerequisites must be checked before choosing microservices?
2. Why does needing independent releases not, by itself, justify microservices?

## The idea in context

An LLM responds from its prompt and training; a Python rule applies the stated condition exactly. Comparing them makes missing context visible.

CI/CD, monitoring, provisioning, and operational ownership cost real effort.

| Key term | Meaning |
| --- | --- |
| **Rule-based system** | A program whose decisions come from rules that people wrote explicitly. |
| **Large language model (LLM)** | A model trained on large amounts of text that continues a prompt with likely text. |
| **Prompt** | The text we send to the model: the instructions, the facts, and the question. |
| **Baseline** | A simple reference result that a more advanced system must beat. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · Write a readiness screen
A project is a dictionary of facts. `decide(project)` returns two values: a decision label and its reason. The two Booleans describe this exercise's stated independent-release goal; they do not replace a full architecture review.

In [ ]:
def decide(project):
    """A simplified readiness screen for these example cases, not a complete design method."""
    ready = project["ops_ready"]
    independent = project["independent_deploys"]
    if ready and independent:
        return "microservices", "independent releases are needed and prerequisites are ready"
    if not ready:
        return "modular monolith", "operational prerequisites are missing"
    return "modular monolith", "a shared release meets the stated need"

pilot = {"name": "ShopEasy pilot", "ops_ready": False, "independent_deploys": False}
print(pilot["name"], decide(pilot))

## Step 2 · Cover all four combinations
Each row below is a separate invented case. `decision, reason = decide(project)` puts the function's two returned values into two named variables. Predict which one case should pass both conditions before running the cell.

In [ ]:
projects = [
    {"name": "ShopEasy pilot", "ops_ready": False, "independent_deploys": False},
    {"name": "Social feeds", "ops_ready": True, "independent_deploys": True},
    {"name": "ShopEasy shared release", "ops_ready": True, "independent_deploys": False},
    {"name": "Returns pilot", "ops_ready": False, "independent_deploys": True},
]
for project in projects:
    decision, reason = decide(project)
    print(project["name"], "->", decision)
    print("  Reason:", reason)

## Step 3 · Normalize a text answer cautiously
A free-text answer is not a typed decision. If it mentions both options or neither, `to_label` returns `unclear`. It cannot interpret arbitrary prose or negation.

In [ ]:
def to_label(reply):
    text = reply.lower()
    if "monolith" in text and "microservice" not in text:
        return "modular monolith"
    if "microservice" in text and "monolith" not in text:
        return "microservices"
    return "unclear"

sample_replies = [
    "Modular Monolith.", "microservices", "Either could work", "Not microservices; a monolith"
]
for reply in sample_replies:
    print(reply, "->", to_label(reply))

## Step 4 · Ask without the screening rule
The model sees the facts but not our exact screen. `ask(prompt, example=...)` returns the example in practice mode; live mode ignores it and makes a real request. `scripted` gives one practice answer per project. Two are deliberately wrong.

The loop counts exact agreement with our screen, not overall architecture quality. We name the raw reply and normalized label separately so you can inspect each.

In [ ]:
scripted = {
    "ShopEasy pilot": "modular monolith",
    "Social feeds": "microservices",
    "ShopEasy shared release": "microservices",
    "Returns pilot": "microservices",
}
agreed = 0
for project in projects:
    prompt = f"Choose modular monolith or microservices for {project}. Reply with the label only."
    reply = ask(prompt, example=scripted[project["name"]])
    answer = to_label(reply)
    expected, reason = decide(project)
    if answer == expected:
        agreed += 1
    print(project["name"], "| screen:", expected, "| model:", answer)
print(f"Agreement: {agreed}/{len(projects)}")

## What you should see
Only Social feeds passes the readiness screen for microservices. The scripted model agrees on 2/4 cases; a live model may differ.

## Try it
1. In Step 2, set `ops_ready=True` in the Returns pilot dictionary, then rerun Steps 2 and 4. Which screen answer changes, and which four prerequisites does this Boolean stand for?
2. In Step 2, set Social feeds to `ops_ready=False`, then rerun Steps 2 and 4. Explain why independent deployment alone is insufficient.
3. Explain why cloning a monolith (X scaling) is different from splitting by function (Y scaling), when deciding how to handle increased traffic.

## Takeaway
Rules are predictable for explicit facts; an LLM can discuss open trade-offs but needs the relevant criteria. Next we put the screen into the prompt and evaluate it.